In [16]:
## 比较不同模型在Adult数据集上的性能

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from fairlearn.metrics import MetricFrame, selection_rate, false_positive_rate, false_negative_rate
from fairlearn.reductions import ExponentiatedGradient, DemographicParity, EqualizedOdds
import math
from scipy.special import softmax
import json

class AdultClient:
    def __init__(self, client_id, data, preprocessor, epsilon=0.1, bin_size=1, sensitive_attribute='sex', is_fair=True, is_private=True, exf=0.1):
        self.client_id = client_id
        self.epsilon = epsilon  
        self.bin_size = bin_size  
        self.is_fair = is_fair  
        self.is_private = is_private  
        self.exf = exf # 对应论文中的公平性阈值 epsilon_f
        self.model = None
        self.original_data = data
        self.protected_data = None
        self.preprocessor = preprocessor
        self.sensitive_attribute = sensitive_attribute
        self.X_train_raw = None
        self.X_test_raw = None
        self.y_train = None
        self.y_test = None
        self.sensitive_train = None
        self.sensitive_test = None
        
    def create_bins(self, data):
        min_sen, max_sen = data[self.sensitive_attribute].min(), data[self.sensitive_attribute].max()
        bins = []
        current = min_sen
        while current < max_sen:
            bins.append((current, current + self.bin_size))
            current += self.bin_size
        bins[-1] = (bins[-1][0], max_sen)
        return bins, len(bins)

    def apply_exponential_mechanism(self):
        """严格按照论文公式实现的隐私保护机制"""
        if not self.is_private:
            self.protected_data = self.original_data.copy()
            return self.protected_data
        
        bins, A_size = self.create_bins(self.original_data)
        self.protected_data = self.original_data.copy()
        # 将敏感属性列转换为浮点数类型，以避免后续赋值时的dtype警告
        self.protected_data[self.sensitive_attribute] = self.protected_data[self.sensitive_attribute].astype(float)
        
        # 计算概率分布 pi (s=a) 和 pi_bar (s!=a)
        pi = math.exp(self.epsilon) / (A_size - 1 + math.exp(self.epsilon))
        pi_bar = 1 / (A_size - 1 + math.exp(self.epsilon))

        for i, row in self.original_data.iterrows():
            a = row[self.sensitive_attribute]
            probs = []
            for interval in bins:
                if interval[0] <= a <= interval[1]:
                    probs.append(pi)
                else:
                    probs.append(pi_bar)
            
            # 归一化并抽样
            probs = np.array(probs) / np.sum(probs)
            selected_idx = np.random.choice(len(bins), p=probs)
            # 使用区间中值作为扰动后的属性
            self.protected_data.at[i, self.sensitive_attribute] = (bins[selected_idx][0] + bins[selected_idx][1]) / 2
        return self.protected_data

    def preprocess_data(self):
        data = self.apply_exponential_mechanism()
        X = data.drop('income', axis=1)
        y = data['income'].map({'<=50K': 0, '>50K': 1})
        
        self.X_train_raw, self.X_test_raw, self.y_train, self.y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        self.X_train = self.preprocessor.transform(self.X_train_raw)
        self.X_test = self.preprocessor.transform(self.X_test_raw)
        self.sensitive_train = self.X_train_raw[self.sensitive_attribute].to_numpy()
        self.sensitive_test = self.X_test_raw[self.sensitive_attribute].to_numpy()

    def train_fedpf_local(self, global_lambda):
        """
        FedPF 客户端逻辑：响应服务端的对偶变量 global_lambda。
        在论文中，Client 是 Learner，其目标是最小化 Lagrangian。
        """
        self.preprocess_data()
        base_model = LogisticRegression(solver='liblinear', max_iter=1000)
        
        if self.is_fair:
            # 论文指出：当全局惩罚 global_lambda 增加时，应加强公平性约束强度
            # 这里通过调整 fairlearn 的 eps 参数来实现自适应
            current_eps = max(0.001, self.exf / (1 + global_lambda))
            self.model = ExponentiatedGradient(base_model, DemographicParity(), eps=current_eps)
            self.model.fit(self.X_train, self.y_train, sensitive_features=self.sensitive_train)
            
            # 计算当前违规量 G(f)，传回给服务端（Auditor）
            y_pred = self.model.predict(self.X_train)
            violation = measure_DP(y_pred, self.sensitive_train)
        else:
            self.model = base_model.fit(self.X_train, self.y_train)
            violation = 0
            
        y_pred_test = self.model.predict(self.X_test)
        acc = accuracy_score(self.y_test, y_pred_test)
        loss = log_loss(self.y_test, y_pred_test)
        eq_odds_diff = max(measure_EO(self.y_test, self.sensitive_test, y_pred_test, 0),measure_EO(self.y_test, self.sensitive_test, y_pred_test, 1))
        return self.model, acc, loss, eq_odds_diff, violation

    def train(self, mode='avg', global_lambda=0.0):
        if mode == 'fedpf':
            return self.train_fedpf_local(global_lambda)
        elif mode == 'ppsgd':
            return self.train_ppsgd()
        elif self.is_fair:
            return self.train_fair_model()
        else:
            # train_base_model() already returns the full tuple, so just return it directly
            return self.train_base_model()
    
    def train_fair_model(self):
        """Train a fairness model"""
        data = self.protected_data if self.is_private and self.protected_data is not None else self.original_data
        
        if data is None:
            if self.is_private:
                self.apply_exponential_mechanism_to_dataset()
                data = self.protected_data
            else:
                data = self.original_data
        
        # Preprocess data
        self.preprocess_data()
        
        # Train a fair model using the ExponentiatedGradient algorithm of fairlearn
        from fairlearn.reductions import ExponentiatedGradient, EqualizedOdds, GridSearch, DemographicParity
        
        # Create fairness constraints
        constraint = EqualizedOdds()
        constraint2 = DemographicParity()
        
        # Create a basic model
        base_model = LogisticRegression(solver='liblinear', random_state=42, max_iter=100)
        
        # Train a fair model using Exponentiated Gradient
        fair_model = ExponentiatedGradient(base_model, constraint2, eps=self.exf)
        # fair_model = GridSearch(base_model, constraint)
        fair_model.fit(self.X_train, self.y_train, sensitive_features=self.sensitive_train)
        
        # Evaluate the model
        y_pred = fair_model.predict(self.X_test)
        accuracy = accuracy_score(self.y_test, y_pred)
        loss = log_loss(self.y_test, y_pred)
        
        # Calculate fairness metrics
        metric_frame = MetricFrame(
            metrics={
                'accuracy': accuracy_score,
                'selection_rate': selection_rate,
                'false_positive_rate': false_positive_rate,
                'false_negative_rate': false_negative_rate,
            },
            y_true=self.y_test,
            y_pred=y_pred,
            sensitive_features=self.sensitive_test
        )
        
        # Calculate the Discrimination
        eq_odds_diff = max(measure_EO(self.y_test, self.sensitive_test, y_pred, 0),measure_EO(self.y_test, self.sensitive_test, y_pred, 1))
        
        print(f"Client {self.client_id} Fair model accuracy: {accuracy:.4f}")
        print(f"Client {self.client_id} Fair model loss: {loss:.4f}")
        print(f"Client {self.client_id} Discrimination: {eq_odds_diff:.4f}")
        
        self.model = fair_model
        
        return self.model, accuracy, loss, eq_odds_diff, 0
    
    def train_base_model(self):
        """Train the basic model"""
        data = self.protected_data if self.is_private and self.protected_data is not None else self.original_data
        
        if data is None:
            if self.is_private:
                self.apply_exponential_mechanism_to_dataset()
                data = self.protected_data
            else:
                data = self.original_data
        
        # Preprocess data
        self.preprocess_data()
        
        # Create and train the model
        self.model = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
        self.model.fit(self.X_train, self.y_train)
        
        # Evaluate the model
        y_pred = self.model.predict(self.X_test)
        accuracy = accuracy_score(self.y_test, y_pred)
        loss = log_loss(self.y_test, y_pred)
        eq_odds_diff = max(measure_EO(self.y_test, self.sensitive_test, y_pred, 0),measure_EO(self.y_test, self.sensitive_test, y_pred, 1))
        
        print(f"Client {self.client_id} base model accuracy: {accuracy:.4f}")
        
        return self.model, accuracy, loss, eq_odds_diff, 0
    
    def train_ppsgd(self):
        """Train with PPSGD style: fine-tune head, then update representation"""
        # Preprocess data
        self.preprocess_data()
        
        # Assume whole model is representation, fine-tune head (full model for linear)
        self.model = LogisticRegression(solver='liblinear', random_state=42, max_iter=100)  # Reduced iter for 'head'
        self.model.fit(self.X_train, self.y_train)
        
        # Then 'update representation' with more iter or full
        self.model.max_iter = 1000
        self.model.fit(self.X_train, self.y_train)  # Continue
        
        y_pred = self.model.predict(self.X_test)
        accuracy = accuracy_score(self.y_test, y_pred)
        loss = log_loss(self.y_test, y_pred)
        eq_odds_diff = max(measure_EO(self.y_test, self.sensitive_test, y_pred, 0),measure_EO(self.y_test, self.sensitive_test, y_pred, 1))
        
        return self.model, accuracy, loss, eq_odds_diff, 0
    
    def evaluate(self, model=None):
        """Evaluate model performance"""
        if model is None:
            model = self.model
        
        if model is None:
            print("The model has not been trained")
            return None
        
        # predict
        y_pred = model.predict(self.X_test)
        
        # Calculate accuracy
        accuracy = accuracy_score(self.y_test, y_pred)
        loss = log_loss(self.y_test, y_pred)
        
        # If fairness is enabled, calculate the fairness metrics.
        eq_odds_diff = max(measure_EO(self.y_test, self.sensitive_test, y_pred, 0),measure_EO(self.y_test, self.sensitive_test, y_pred, 1))
        return accuracy, loss, eq_odds_diff

# ================= 2. 核心度量函数 =================
def measure_EO(Y, A, Yhat, y_value):
    unique_groups = np.unique(A)
    expectations = []
    for group in unique_groups:
        mask = (Y == y_value) & (A == group)
        if np.sum(mask) > 0:
            expectations.append(np.mean(Yhat[mask]))
    return np.max(expectations) - np.min(expectations) if len(expectations) > 1 else 0

def measure_DP(Yhat, A):
    unique_groups = np.unique(A)
    expectations = [np.mean(Yhat[A == group]) for group in unique_groups]
    return np.max(expectations) - np.min(expectations) if len(expectations) > 1 else 0

# ================= 3. 聚合与博弈逻辑 =================
def federated_aggregation(clients, mode='avg', epsilon=0.1, rank_ratio=0.5, temp=1.0, local_eq_odds_diffs=None, current_lambda=0.0, lr_lambda=0.1, current_round=0):
    """
    实现论文 Algorithm 1 中的聚合与对偶更新逻辑。
    """
    coefs, intercepts, violations = [], [], []
    
    for client in clients:
        # 提取参数
        if hasattr(client.model, 'predictors_'):
            w = np.array(client.model.weights_)
            c = np.sum([p.coef_[0] * w[i] for i, p in enumerate(client.model.predictors_)], axis=0)
            inter = np.sum([p.intercept_[0] * w[i] for i, p in enumerate(client.model.predictors_)])
        else:
            c, inter = client.model.coef_[0], client.model.intercept_[0]
        coefs.append(c)
        intercepts.append(inter)
        # 计算违规
        y_pred = client.model.predict(client.X_train)
        violations.append(measure_DP(y_pred, client.sensitive_train))

    # 聚合参数
    avg_coef = np.mean(coefs, axis=0)
    avg_intercept = np.mean(intercepts)
    avg_violation = np.mean(violations)

    # 服务端更新对偶变量 (Auditor 逻辑: 极大化违规惩罚)
    # lambda = lambda + lr * (Violation - epsilon_f)
    # 这里 exf 是预设的公平性容忍度
    new_lambda = max(0, current_lambda + lr_lambda * (avg_violation - clients[0].exf))

    # 构建全局模型
    global_model = LogisticRegression(solver='liblinear')
    global_model.coef_ = np.array([avg_coef])
    global_model.intercept_ = np.array([avg_intercept])
    global_model.classes_ = np.array([0, 1])
    global_model.preprocessor = clients[0].preprocessor
    
    if mode == 'fedpf':
        return global_model, new_lambda
    else:
        return global_model, 0.0  # For non-fedpf

# ================= 4. 运行框架 =================
def run_federated_learning(num_clients=3, num_rounds=5, epsilon=0.1, bin_size=1, sensitive_attribute='sex', is_fair=True, is_private=True, exf=0.1, mode='avg', rank_ratio=0.5, temp=1.0):
    """Run the FL training process with mode for different algorithms"""
    # Load all data and fit global preprocessor
    columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 
               'marital-status', 'occupation', 'relationship', 'race', 'sex', 
               'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
    all_data = pd.read_csv('../dataset/adult/train.csv', names=columns, skipinitialspace=True, na_values='?', nrows=20000)
    all_data['sex'] = all_data['sex'].map({'Male': 0, 'Female': 1})
    all_data = all_data.dropna()
    
    X_all = all_data.drop('income', axis=1)
    numeric_features = X_all.select_dtypes(include=['int64', 'float64']).columns
    categorical_features = X_all.select_dtypes(include=['object']).columns
    
    global_preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
        ])
    global_preprocessor.fit(X_all)
    
    # Split data for clients
    n = len(all_data)
    client_datas = [
        all_data.iloc[:n//3],
        all_data.iloc[n//3:2*n//3],
        all_data.iloc[2*n//3:]
    ]
    
    clients = []
    for i in range(num_clients):
        client = AdultClient(client_id=i, data=client_datas[i], preprocessor=global_preprocessor, epsilon=epsilon, bin_size=bin_size, sensitive_attribute=sensitive_attribute, is_fair=is_fair, is_private=is_private, exf=exf)
        clients.append(client)
    
    # Record the metrices in each round
    global_accuracies = []
    global_losses = []
    global_eq_odds_diffs = []
    
    # Init global λ for FedPF
    global_lambda = 0.0
    
    # The training process of federated learning
    for round in range(num_rounds):
        print(f"\n===== Federated learning rounds {round+1}/{num_rounds} ({mode}) =====")
        
        # Local training
        local_models = []
        local_accuracies = []
        local_losses = []
        local_eq_odds_diffs = []
        local_violations = []
        
        for client in clients:
            print(f"\n Client {client.client_id} local training in progress...")
            if mode == 'fedpf':
                model, accuracy, loss, eq_odds_diff, violation = client.train(mode=mode, global_lambda=global_lambda)
                local_violations.append(violation)
            else:
                # client.train() now returns 5 values for all modes
                model, accuracy, loss, eq_odds_diff, _ = client.train(mode=mode)
                violation = 0
            local_models.append(model)
            local_accuracies.append(accuracy)
            local_losses.append(loss)
            if eq_odds_diff is not None:
                local_eq_odds_diffs.append(eq_odds_diff)
        
        # Model Aggregation
        print("\nModel aggregation in progress...")
        global_model, global_lambda = federated_aggregation(clients, mode=mode, epsilon=epsilon, rank_ratio=rank_ratio, temp=temp, local_eq_odds_diffs=local_eq_odds_diffs, current_lambda=global_lambda, current_round=round)
        
        # Evaluate the global model
        global_accuracy = 0
        global_loss = 0
        global_eq_odds_diff = 0
        
        for client in clients:
            print(f"Evaluation client {client.client_id} of the model...")
            
            # Use the preprocessor of the global model to process client data
            X_client_test = client.X_test_raw
            y_client_test = client.y_test
            
            # Use the pre - processor of the global model to process the client test data
            X_client_test_processed = global_model.preprocessor.transform(X_client_test)
            
            # Use the global model for prediction
            y_pred = global_model.predict(X_client_test_processed)
            # Calculate accuracy
            accuracy = accuracy_score(y_client_test, y_pred)
            loss = log_loss(y_client_test, y_pred)
            print(f"global server accuracy:------------{accuracy}")
            print(f"global server loss:------------{loss}")
            
            # If fairness is enabled, calculate the fairness metrics.
            
            sensitive_client_test = X_client_test[client.sensitive_attribute]
            eq_odds_diff = max(measure_EO(y_client_test, sensitive_client_test, y_pred, 0),measure_EO(y_client_test, sensitive_client_test, y_pred, 1))

            print(f"Accuracy: {accuracy:.4f}, loss: {loss:.4f}, Discrimination: {eq_odds_diff:.4f}")
            global_accuracy += accuracy
            global_loss += loss
            global_eq_odds_diff += eq_odds_diff

        
        global_accuracy /= num_clients
        global_loss /= num_clients
        global_eq_odds_diff /= num_clients
        print(f"\nGlobal model performance: Accuracy = {global_accuracy:.4f}, Loss = {global_loss:.4f}, Discrimination = {global_eq_odds_diff:.4f}")
      
        
        # Save the results of this round
        global_accuracies.append(global_accuracy)
        global_losses.append(global_loss)
        global_eq_odds_diffs.append(global_eq_odds_diff)
        
        # Distribute the global model to each client
        for client in clients:
            client.model = global_model
            client.old_model = global_model  # For PPSGD diff
        
    # Return the final model and performance metrics
    return global_model, global_accuracies, global_losses, global_eq_odds_diffs


# Analyze the privacy-fairness-utility trade-off
eps = [-2, -1, 0, 1, 2] # log(epsilon)
sensitive_attribute = 'age' 
results = {}

for ep in eps:
    epsilon = 10**ep
    print(f"\n===== Privacy budget = {epsilon} =====")
    
    # FedAvg (no fair, no private, avg mode)
    print("\n--- FedAvg ---")
    _, accuracies_fedavg, losses_fedavg, eq_odds_diffs_fedavg = run_federated_learning(
        num_clients=3, 
        num_rounds=1, 
        epsilon=epsilon, 
        is_fair=True,
        is_private=True,
        bin_size=4,
        sensitive_attribute=sensitive_attribute,
        exf=0.1,
        mode='avg'
    )
    
    # FedPF (fair, private, avg mode)
    print("\n--- FedPF (Ours) ---")
    _, accuracies_fedpf, losses_fedpf, eq_odds_diffs_fedpf = run_federated_learning(
        num_clients=3, 
        num_rounds=1, 
        epsilon=epsilon, 
        is_fair=True,
        is_private=True,
        bin_size=4,
        sensitive_attribute=sensitive_attribute,
        exf=0.1,
        mode='avg'
    )

    # FedAA (no fair, no private, aa mode for adaptive fairness)
    print("\n--- FedAA ---")
    _, accuracies_fedaa, losses_fedaa, eq_odds_diffs_fedaa = run_federated_learning(
        num_clients=3, 
        num_rounds=1, 
        epsilon=epsilon, 
        is_fair=True,
        is_private=True,
        bin_size=4,
        sensitive_attribute=sensitive_attribute,
        exf=0.1,
        mode='aa',
        temp=1.0
    )

    # CENTAUR (no fair, private, centaur mode with param noise)
    print("\n--- CENTAUR ---")
    _, accuracies_centaur, losses_centaur, eq_odds_diffs_centaur = run_federated_learning(
        num_clients=3, 
        num_rounds=1, 
        epsilon=epsilon, 
        is_fair=True,
        is_private=True,
        bin_size=4,
        sensitive_attribute=sensitive_attribute,
        exf=0.1,
        mode='centaur'
    )

    # FedCEO (no fair, private, ceo mode with low-rank)
    print("\n--- FedCEO ---")
    _, accuracies_fedceo, losses_fedceo, eq_odds_diffs_fedceo = run_federated_learning(
        num_clients=3, 
        num_rounds=1, 
        epsilon=epsilon, 
        is_fair=True,
        is_private=True,
        bin_size=4,
        sensitive_attribute=sensitive_attribute,
        exf=0.1,
        mode='ceo',
        rank_ratio=0.5
    )
    
    # PPSGD (no fair, private, ppsgd mode with diff aggregate)
    print("\n--- PPSGD ---")
    _, accuracies_ppsgd, losses_ppsgd, eq_odds_diffs_ppsgd = run_federated_learning(
        num_clients=3, 
        num_rounds=1, 
        epsilon=epsilon, 
        is_fair=True,
        is_private=True,
        bin_size=4,
        sensitive_attribute=sensitive_attribute,
        exf=0.1,
        mode='ppsgd'
    )
    
    results[ep] = {
        'fedavg': {
            'accuracy': accuracies_fedavg[-1],
            'loss': losses_fedavg[-1],
            'eq_odds_diff': eq_odds_diffs_fedavg[-1]
            },
        'fedpf': {
            'accuracy': accuracies_fedpf[-1],
            'loss': losses_fedpf[-1],
            'eq_odds_diff': eq_odds_diffs_fedpf[-1]
            },
        'fedaa': {
            'accuracy': accuracies_fedaa[-1],
            'loss': losses_fedaa[-1],
            'eq_odds_diff': eq_odds_diffs_fedaa[-1]
            },
        'centaur': {
            'accuracy': accuracies_centaur[-1],
            'loss': losses_centaur[-1],
            'eq_odds_diff': eq_odds_diffs_centaur[-1]
            },
        'fedceo': {
            'accuracy': accuracies_fedceo[-1],
            'loss': losses_fedceo[-1],
            'eq_odds_diff': eq_odds_diffs_fedceo[-1]
            },
        'ppsgd': {
            'accuracy': accuracies_ppsgd[-1],
            'loss': losses_ppsgd[-1],
            'eq_odds_diff': eq_odds_diffs_ppsgd[-1]
            }
    }

# You can add plotting code here to compare results across epsilons and algorithms
import json
print(json.dumps(results, indent=4))


===== Privacy budget = 0.01 =====

--- FedAvg ---

===== Federated learning rounds 1/1 (avg) =====

 Client 0 local training in progress...
Client 0 Fair model accuracy: 0.8066
Client 0 Fair model loss: 6.9696
Client 0 Discrimination: 0.5333

 Client 1 local training in progress...
Client 1 Fair model accuracy: 0.8042
Client 1 Fair model loss: 7.0571
Client 1 Discrimination: 0.4566

 Client 2 local training in progress...
Client 2 Fair model accuracy: 0.8293
Client 2 Fair model loss: 6.1531
Client 2 Discrimination: 0.7841

Model aggregation in progress...
Evaluation client 0 of the model...
global server accuracy:------------0.8317152103559871
global server loss:------------6.065598628589294
Accuracy: 0.8317, loss: 6.0656, Discrimination: 0.5143
Evaluation client 1 of the model...
global server accuracy:------------0.8349514563106796
global server loss:------------5.948952501116423
Accuracy: 0.8350, loss: 5.9490, Discrimination: 0.3982
Evaluation client 2 of the model...
global server

In [7]:
import json
import pandas as pd
import matplotlib.pyplot as plt

# 替换为您的JSON数据
data = {
    "-2": {
        "fedavg": {
            "accuracy": 0.8357605177993527,
            "loss": 5.919790969248205,
            "eq_odds_diff": 1.0
        },
        "fedpf": {
            "accuracy": 0.8311758360302051,
            "loss": 6.085039649834773,
            "eq_odds_diff": 1.0
        },
        "fedaa": {
            "accuracy": 0.8357605177993527,
            "loss": 5.919790969248205,
            "eq_odds_diff": 1.0
        },
        "centaur": {
            "accuracy": 0.54638619201726,
            "loss": 16.349898867447422,
            "eq_odds_diff": 1.0
        },
        "fedceo": {
            "accuracy": 0.8309061488673138,
            "loss": 6.094760160457512,
            "eq_odds_diff": 1.0
        }
    },
    "-1": {
        "fedavg": {
            "accuracy": 0.8357605177993527,
            "loss": 5.919790969248205,
            "eq_odds_diff": 1.0
        },
        "fedpf": {
            "accuracy": 0.808252427184466,
            "loss": 6.911283052767609,
            "eq_odds_diff": 0.8787878787878788
        },
        "fedaa": {
            "accuracy": 0.8357605177993527,
            "loss": 5.919790969248205,
            "eq_odds_diff": 1.0
        },
        "centaur": {
            "accuracy": 0.7464940668824164,
            "loss": 9.137279985374898,
            "eq_odds_diff": 1.0
        },
        "fedceo": {
            "accuracy": 0.8309061488673138,
            "loss": 6.094760160457511,
            "eq_odds_diff": 1.0
        }
    },
    "0": {
        "fedavg": {
            "accuracy": 0.8357605177993527,
            "loss": 5.919790969248205,
            "eq_odds_diff": 1.0
        },
        "fedpf": {
            "accuracy": 0.8276699029126213,
            "loss": 6.211406287930383,
            "eq_odds_diff": 1.0
        },
        "fedaa": {
            "accuracy": 0.8357605177993527,
            "loss": 5.919790969248205,
            "eq_odds_diff": 1.0
        },
        "centaur": {
            "accuracy": 0.7001078748651564,
            "loss": 10.80920781248605,
            "eq_odds_diff": 1.0
        },
        "fedceo": {
            "accuracy": 0.8303667745415318,
            "loss": 6.114201181702991,
            "eq_odds_diff": 1.0
        }
    },
    "1": {
        "fedavg": {
            "accuracy": 0.8357605177993527,
            "loss": 5.919790969248205,
            "eq_odds_diff": 1.0
        },
        "fedpf": {
            "accuracy": 0.7820927723840345,
            "loss": 7.854172583173317,
            "eq_odds_diff": 0.7999999999999999
        },
        "fedaa": {
            "accuracy": 0.8357605177993527,
            "loss": 5.919790969248205,
            "eq_odds_diff": 1.0
        },
        "centaur": {
            "accuracy": 0.8349514563106797,
            "loss": 5.948952501116423,
            "eq_odds_diff": 1.0
        },
        "fedceo": {
            "accuracy": 0.8357605177993527,
            "loss": 5.919790969248207,
            "eq_odds_diff": 1.0
        }
    },
    "2": {
        "fedavg": {
            "accuracy": 0.8357605177993527,
            "loss": 5.919790969248205,
            "eq_odds_diff": 1.0
        },
        "fedpf": {
            "accuracy": 0.784789644012945,
            "loss": 7.756967476945924,
            "eq_odds_diff": 0.8333333333333334
        },
        "fedaa": {
            "accuracy": 0.8357605177993527,
            "loss": 5.919790969248205,
            "eq_odds_diff": 1.0
        },
        "centaur": {
            "accuracy": 0.837108953613808,
            "loss": 5.871188416134509,
            "eq_odds_diff": 1.0
        },
        "fedceo": {
            "accuracy": 0.8362998921251349,
            "loss": 5.900349948002727,
            "eq_odds_diff": 1.0
        }
    }
}

rows = []
for log_eps, algos in data.items():
    for algo, metrics in algos.items():
        row = {'log_epsilon': float(log_eps), 'algorithm': algo, **metrics}
        rows.append(row)

df = pd.DataFrame(rows)
pivot_df = df.pivot(index='log_epsilon', columns='algorithm', values=['accuracy', 'loss', 'eq_odds_diff'])
print(pivot_df)

summary = df.groupby('algorithm').agg({'accuracy': ['mean', 'std'], 'loss': ['mean', 'std'], 'eq_odds_diff': ['mean', 'std']})
print(summary)

# 绘图代码类似上述，循环算法绘制线图

             accuracy                                               loss  \
algorithm     centaur     fedaa    fedavg    fedceo     fedpf    centaur   
log_epsilon                                                                
-2.0         0.546386  0.835761  0.835761  0.830906  0.831176  16.349899   
-1.0         0.746494  0.835761  0.835761  0.830906  0.808252   9.137280   
 0.0         0.700108  0.835761  0.835761  0.830367  0.827670  10.809208   
 1.0         0.834951  0.835761  0.835761  0.835761  0.782093   5.948953   
 2.0         0.837109  0.835761  0.835761  0.836300  0.784790   5.871188   

                                                    eq_odds_diff               \
algorithm       fedaa    fedavg    fedceo     fedpf      centaur fedaa fedavg   
log_epsilon                                                                     
-2.0         5.919791  5.919791  6.094760  6.085040          1.0   1.0    1.0   
-1.0         5.919791  5.919791  6.094760  6.911283          1.0   